# 04 · SciPy 루틴 (cupyx.scipy.*)

> **CuPy 2일 집중 코스 — Day 1 / 단원 2 (NumPy/SciPy CuPy 프로그래밍)**

공식 [overview](https://docs.cupy.dev/en/stable/overview.html)의 **SciPy Routines** 를 본격적으로 — `scipy.*` → `cupyx.scipy.*` 치환으로.
FFT·선형대수·이미지 처리·**희소행렬**·희소 선형대수·신호처리를 미니앱과 함께 다룹니다.

## 학습 목표
- `scipy.*` 코드를 `cupyx.scipy.*` 로 포팅하고 CPU와 정확성을 검증한다.
- 희소행렬 포맷·내부구조를 이해하고 2D 푸아송을 직접/반복법으로 푼다.
- 이미지 분석(`ndimage`)·신호 필터(`signal`)·수치 안정 함수(`special`)를 활용한다.

## 목차
1. [FFT `scipy.fft` + DCT 압축](#1)
2. [선형대수 `scipy.linalg`](#2)
3. [이미지 처리 `ndimage` + 연결요소](#3)
4. [희소행렬 `sparse` (+포맷·내부구조)](#4)
5. [희소 선형대수 `sparse.linalg` + 2D 푸아송](#5)
6. [신호처리 `signal`](#6)
7. [special · stats](#7)
8. [체크포인트](#8)

> 백엔드: cuFFT·cuSOLVER·cuSPARSE. 모두 `scipy`와 동일 인터페이스입니다.

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, cpu_ms, gpu_ms, compare, allclose
print_env()

<a id="1"></a>
## 1. FFT `scipy.fft` + DCT 압축

📖 [`cupyx.scipy.fft`](https://docs.cupy.dev/en/stable/reference/scipy_fft.html) — `fft`, `rfft`, `dct/idct`, `dst`, `next_fast_len`

### 🔬 이론 배경 — DCT
- **DCT**(이산 코사인 변환): 실수 코사인 기저로 변환. 신호 에너지를 **소수의 저주파 계수**에 집중시킴.
- → 상위 계수만 남기면 **손실 압축**(JPEG·MP3의 핵심 원리).

In [ ]:
from scipy.fft import dct as sp_dct, idct as sp_idct
from cupyx.scipy.fft import dct as cp_dct, idct as cp_idct, next_fast_len
x_np = np.random.random(100_000).astype(np.float32); x_cp = cp.asarray(x_np)
allclose(sp_dct(x_np, norm='ortho'), cp_dct(x_cp, norm='ortho'), rtol=1e-3, atol=1e-3, name='DCT')
print('next_fast_len(100000)=', next_fast_len(100_000))

### 미니앱 — DCT 기반 압축
DCT 계수의 **상위 일부만 유지**하고 역변환하면 손실 압축이 됩니다. 유지 비율에 따른 복원 오차를 보세요.

In [ ]:
def dct_compress(x, keep):
    xp = cp.get_array_module(x)
    dct = cp_dct if xp is cp else sp_dct
    idct = cp_idct if xp is cp else sp_idct
    X = dct(x, norm='ortho'); X[keep:] = 0
    return idct(X, norm='ortho')

t = np.linspace(0,1,4096,endpoint=False).astype(np.float32)
sig = (np.sin(2*np.pi*3*t)+0.3*np.sin(2*np.pi*40*t)).astype(np.float32)
for keep in [64, 256, 1024]:
    r = dct_compress(cp.asarray(sig), keep)
    err = float(cp.linalg.norm(r-cp.asarray(sig))/cp.linalg.norm(cp.asarray(sig)))
    print(f'keep={keep:>5}  상대오차={err:.3e}')

<a id="2"></a>
## 2. 선형대수 `scipy.linalg`

📖 [`cupyx.scipy.linalg`](https://docs.cupy.dev/en/stable/reference/scipy_linalg.html) — `lu_factor/lu_solve`, `solve_triangular`, `expm`, `toeplitz` 등

In [ ]:
from cupyx.scipy.linalg import lu_factor, lu_solve, expm
n=1200
A = cp.random.random((n,n),dtype=cp.float32) + n*cp.eye(n,dtype=cp.float32)
b = cp.random.random(n,dtype=cp.float32)
lu,piv = lu_factor(A); x = lu_solve((lu,piv), b)
allclose(A@x, b, rtol=1e-3, atol=1e-3, name='lu_solve')

**연습 — 행렬지수 `expm`**: `scipy.linalg.expm` 코드를 GPU로 포팅하고 CPU와 비교하세요.

In [ ]:
from scipy.linalg import expm as sp_expm
from cupyx.scipy.linalg import expm as cp_expm
M_np = (0.1*np.random.randn(64,64)).astype(np.float32)
# TODO: allclose(sp_expm(M_np), cp.asnumpy(cp_expm(cp.asarray(M_np))), rtol=1e-2, atol=1e-2, name='expm')

<details><summary>💡 해답 보기</summary>

```python
allclose(sp_expm(M_np), cp.asnumpy(cp_expm(cp.asarray(M_np))), rtol=1e-2, atol=1e-2, name='expm')
```
</details>

<a id="3"></a>
## 3. 이미지 처리 `ndimage` + 연결요소

📖 [`cupyx.scipy.ndimage`](https://docs.cupy.dev/en/stable/reference/scipy_ndimage.html) — `gaussian_filter`, `sobel`, `label`, `center_of_mass`, `rotate`, `zoom`

### 🔬 이론 배경 — 컨볼루션 필터
- 작은 **커널**을 이미지 위로 슬라이딩하며 이웃 픽셀의 **가중합**(컨볼루션).
- **가우시안** = 평활(잡음 제거), **소벨** = 공간 미분(엣지 검출).
- **`label`** = 연결요소(서로 인접한 전경 픽셀 그룹) 식별.

In [ ]:
import scipy.ndimage as spnd
import cupyx.scipy.ndimage as cpnd
# 합성 이미지: 밝은 원반 5개 + 잡음
rng=np.random.default_rng(0); H=W=256
img=np.zeros((H,W),np.float32)
centers=[(60,60),(60,190),(190,60),(190,190),(128,128)]
yy,xx=np.mgrid[0:H,0:W]
for cy,cx in centers: img += np.exp(-(((yy-cy)**2+(xx-cx)**2)/200.0)).astype(np.float32)
img += 0.05*rng.standard_normal((H,W)).astype(np.float32)

def count_blobs(image, ndi):
    g = ndi.gaussian_filter(image, sigma=2)
    mask = g > 0.5
    lab, num = ndi.label(mask)
    return num
print('연결요소 CPU:', count_blobs(img, spnd), '| GPU:', count_blobs(cp.asarray(img), cpnd))

**연습 — 그래디언트 크기**: 블러 후 `sobel`로 x·y 그래디언트 크기를 구하는 장치 비종속 함수를 완성하세요.

In [ ]:
def grad_magnitude(image):
    # TODO: 모듈 선택(spnd/cpnd) -> gaussian_filter(sigma=2) -> sobel(axis=0/1) -> sqrt(gx^2+gy^2)
    raise NotImplementedError
# ref=grad_magnitude(img); out=cp.asnumpy(grad_magnitude(cp.asarray(img)))
# allclose(ref,out,rtol=1e-3,atol=1e-3,name='grad_mag')

<details><summary>💡 해답 보기</summary>

```python
def grad_magnitude(image):
    if cp.get_array_module(image) is cp:
        ndi = cpnd; xp = cp
    else:
        ndi = spnd; xp = np
    g = ndi.gaussian_filter(image, sigma=2)
    gx = ndi.sobel(g, axis=0); gy = ndi.sobel(g, axis=1)
    return xp.sqrt(gx*gx + gy*gy)

ref = grad_magnitude(img); out = cp.asnumpy(grad_magnitude(cp.asarray(img)))
allclose(ref, out, rtol=1e-3, atol=1e-3, name='grad_mag')
```
</details>

<a id="4"></a>
## 4. 희소행렬 `sparse` (+ 포맷·내부구조)

📖 [`cupyx.scipy.sparse`](https://docs.cupy.dev/en/stable/reference/scipy_sparse.html) (cuSPARSE)

0이 아닌 값만 저장하는 2-D 포맷 네 가지를 제공합니다.

| 포맷 | 클래스 | 특징 |
|------|--------|------|
| 압축 행 | `csr_matrix` | 행 압축 — **SpMV 효율적** |
| 좌표 | `coo_matrix` | `(data,(row,col))` 구성 편리 |
| 압축 열 | `csc_matrix` | 열 압축 |
| 대각 | `dia_matrix` | 대각 위주 |

CSR은 `data`(값), `indices`(열 번호), `indptr`(행 시작 위치) 세 배열로 저장됩니다.

### 🔬 이론 배경 — 희소행렬은 어떻게 저장되나 (CSR)
- **CSR**은 3개 배열로 저장:
  - `data`: 0이 아닌 값들(행 우선 순서)
  - `indices`: 각 값의 **열 번호**
  - `indptr`: 각 **행의 시작 오프셋**(길이 = 행수+1, 마지막 = 총 nnz)
- 행 i의 원소 = `data[indptr[i]:indptr[i+1]]`, 열 = `indices[동일 구간]`.
- 메모리는 **O(nnz)**(비영 개수)뿐 — 밀집 저장 O(n²)보다 훨씬 작고, 행 순회·SpMV에 효율적. (출처: SciPy `csr_matrix`)

In [ ]:
import scipy.sparse as sps
import cupyx.scipy.sparse as cps
# COO -> CSR 내부 구조 관찰
rows=cp.array([0,0,1,2]); cols=cp.array([0,2,1,2]); vals=cp.array([4.,1.,3.,2.],dtype=cp.float32)
A = cps.coo_matrix((vals,(rows,cols)), shape=(3,3)).tocsr()
print('dense=\n', cp.asnumpy(A.toarray()))
print('data   :', cp.asnumpy(A.data))
print('indices:', cp.asnumpy(A.indices))
print('indptr :', cp.asnumpy(A.indptr))

In [ ]:
# 2D 라플라시안을 diags+kron 으로 구성 (SciPy/CuPy 동일 코드)
def laplacian_2d(S, n):
    I = S.identity(n, format='csr', dtype='float32')
    T = S.diags([-1.,2.,-1.], [-1,0,1], shape=(n,n), format='csr', dtype='float32')
    return (S.kron(I,T) + S.kron(T,I)).tocsr()
n=60; A_cpu=laplacian_2d(sps,n); A_gpu=laplacian_2d(cps,n)
x_np=np.random.random(n*n).astype(np.float32); x_cp=cp.asarray(x_np)
allclose(A_cpu@x_np, A_gpu@x_cp, rtol=1e-4, atol=1e-4, name='SpMV')
compare('SpMV', lambda:A_cpu@x_np, lambda:A_gpu@x_cp, n_repeat=20, n_warmup=3)

<a id="5"></a>
## 5. 희소 선형대수 `sparse.linalg` + 2D 푸아송

📖 [`cupyx.scipy.sparse.linalg`](https://docs.cupy.dev/en/stable/reference/scipy_sparse_linalg.html) — `cg`, `gmres`, `spsolve`, `eigsh`, `svds`, `norm`

### 🔬 이론 배경 — 직접법 vs 반복법
- **직접법**(`spsolve`, LU): 분해로 정확해를 구하지만, 큰 희소행렬은 **fill-in**으로 메모리 부담.
- **반복법**(`cg` 등): 행렬-벡터 곱(SpMV)만 반복해 근사해에 수렴. 메모리 적음.
- **CG**는 **대칭 양의 정부호(SPD)** 행렬에서 잔차를 최소화하며 수렴 — 라플라시안이 대표 예.

In [ ]:
from cupyx.scipy.sparse.linalg import cg, spsolve, eigsh
b = cp.random.random(n*n, dtype=cp.float32)
# 직접법(spsolve) vs 반복법(cg)
x_cg, info = cg(A_gpu, b, maxiter=2000, rtol=1e-6)
x_dir = spsolve(A_gpu.tocsr(), b)
print('cg info', info, '| cg vs spsolve 상대차:', float(cp.linalg.norm(x_cg-x_dir)/cp.linalg.norm(x_dir)))
vals = eigsh(A_gpu, k=3, which='LM', return_eigenvectors=False)
print('최대 고유값 3개:', cp.asnumpy(cp.sort(vals))[::-1])

**연습 — 1D 라플라시안 + CG**: `diags`로 1D 라플라시안(2,-1)을 만들고 `cg`로 푸세요.

In [ ]:
def laplacian_1d(n):
    # TODO: cps.diags([-1,2,-1],[-1,0,1],shape=(n,n),format='csr',dtype='float32')
    raise NotImplementedError
# n=5000; A=laplacian_1d(n); b=cp.random.random(n,dtype=cp.float32)
# sol,info=cg(A,b,maxiter=5000,rtol=1e-5); print(info, float(cp.linalg.norm(A@sol-b)/cp.linalg.norm(b)))

<details><summary>💡 해답 보기</summary>

```python
def laplacian_1d(n):
    return cps.diags([-1.,2.,-1.],[-1,0,1],shape=(n,n),format='csr',dtype='float32')
```
</details>

<a id="6"></a>
## 6. 신호처리 `signal`

📖 [`cupyx.scipy.signal`](https://docs.cupy.dev/en/stable/reference/scipy_signal.html) — `fftconvolve`, `butter`+`filtfilt`, `spectrogram`, `welch` 등

In [ ]:
from scipy.signal import butter as spb, filtfilt as spf
from cupyx.scipy.signal import butter as cpb, filtfilt as cpf, spectrogram
t=np.linspace(0,1,20000,endpoint=False).astype(np.float32)
sig_np=(np.sin(2*np.pi*5*t)+0.4*np.random.randn(t.size)).astype(np.float32)
def lowpass(xb,xf,sig):
    b,a=xb(4,0.05); return xf(b,a,sig)
allclose(lowpass(spb,spf,sig_np), cp.asnumpy(lowpass(cpb,cpf,cp.asarray(sig_np))), rtol=1e-2, atol=1e-2, name='butter+filtfilt')
# 스펙트로그램(처프)
f,tt,Sxx = spectrogram(cp.asarray(sig_np), fs=20000)
print('spectrogram shape:', Sxx.shape)

<a id="7"></a>
## 7. special · stats

📖 [`cupyx.scipy.special`](https://docs.cupy.dev/en/stable/reference/scipy_special.html) · [`cupyx.scipy.stats`](https://docs.cupy.dev/en/stable/reference/scipy_stats.html)

In [ ]:
from cupyx.scipy.special import softmax, logsumexp, expit, erf
from cupyx.scipy.stats import zscore, trim_mean
v = cp.linspace(-6,6,7,dtype=cp.float32)
print('expit:', cp.asnumpy(expit(v)).round(3))
print('softmax 합:', float(softmax(v).sum()))
data = cp.random.random(1_000_000, dtype=cp.float32)
print('zscore mean/std:', float(zscore(data).mean()), float(zscore(data).std()))
print('trim_mean(10%):', float(trim_mean(data, 0.1)))

**연습 — 수치 안정 softmax**: `logsumexp`로 직접 softmax를 구현하고 `special.softmax`와 비교하세요.

In [ ]:
def my_softmax(x):
    # TODO: xp.exp(x - logsumexp(x))
    raise NotImplementedError
# z=cp.random.random(1000,dtype=cp.float32)*20
# allclose(my_softmax(z), softmax(z), rtol=1e-4, atol=1e-6, name='softmax')

<details><summary>💡 해답 보기</summary>

```python
def my_softmax(x):
    return cp.exp(x - logsumexp(x))
z = cp.random.random(1000, dtype=cp.float32)*20
allclose(my_softmax(z), softmax(z), rtol=1e-4, atol=1e-6, name='softmax')
```
</details>

<a id="8"></a>
## 8. 체크포인트

- [ ] `scipy.fft`(DCT)로 압축 데모를 했다
- [ ] `scipy.linalg`(lu_solve·expm)를 GPU에서 실행했다
- [ ] `ndimage`로 연결요소 라벨링·그래디언트를 구했다
- [ ] 희소 CSR 내부구조(data/indices/indptr)를 이해하고 SpMV했다
- [ ] `sparse.linalg`로 2D 푸아송을 직접/반복법으로 풀었다
- [ ] `signal`(butter+filtfilt·spectrogram)·`special`(안정 softmax)을 사용했다

**Day 1 단원 2 완료.** 다음: **`05_memory_profiling`**.